In [1]:
import json
import pandas as pd
from IPython.display import HTML, display

# Load PoC results generated by POC.ipynb
df = pd.read_csv("../data/results/poc_gemma3_results.csv")

# Prepare clean JSON payload for the interactive notebook view
records = []
for _, row in df.iterrows():
    records.append({
        "item_id": int(row["item_id"]),
        "gold_label": int(row["gold_label"]),
        "majority_decision": int(row["majority_decision"]),
        "self_consistency": float(row["self_consistency"]),
        "conf": float(row["mean_verbalised_conf"]),
        "is_correct": bool(row["is_correct"])
    })

data_json = json.dumps(records)

html_template = f"""
<div id="poc-dashboard" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; background: #ffffff; color: #1e293b; padding: 18px; border-radius: 10px; border: 1px solid #cbd5e1; box-shadow: 0 2px 8px rgba(0,0,0,0.06); max-width: 1280px;">
    
    <!-- Top Control Bar: Auto-Play Button & Interactive Threshold Slider -->
    <div style="display: flex; align-items: center; justify-content: space-between; background: #f8fafc; padding: 12px 18px; border-radius: 8px; border: 1px solid #e2e8f0; margin-bottom: 16px; gap: 16px;">
        <div style="display: flex; align-items: center; gap: 12px;">
            <button id="play-btn" style="background: #1d4ed8; color: white; border: none; padding: 8px 16px; border-radius: 6px; font-weight: 600; cursor: pointer; font-size: 13px; min-width: 135px;">
                ⏸ Pause Auto-Play
            </button>
            <span style="font-size: 13px; color: #475569;">Drag slider to override manually:</span>
        </div>
        
        <div style="flex-grow: 1; display: flex; align-items: center; gap: 12px;">
            <span style="font-size: 12px; font-weight: 600; color: #64748b;">0.80</span>
            <input id="thresh-slider" type="range" min="0.80" max="0.98" step="0.01" value="0.82" style="flex-grow: 1; cursor: pointer; accent-color: #1d4ed8; height: 6px;">
            <span style="font-size: 12px; font-weight: 600; color: #64748b;">0.98</span>
        </div>
        
        <div style="background: #eff6ff; border: 1px solid #bfdbfe; padding: 6px 14px; border-radius: 6px; min-width: 195px; text-align: center;">
            <span style="font-size: 12px; color: #1e40af; font-weight: 600;">Acceptance Threshold: </span>
            <span id="thresh-value" style="font-size: 16px; font-weight: 800; color: #1d4ed8;">0.82</span>
        </div>
    </div>

    <!-- 3 Synchronized Panels Grid -->
    <div style="display: grid; grid-template-columns: 1.25fr 1.05fr 1.15fr; gap: 16px; align-items: stretch;">
        
        <!-- PANEL 1: Per-Article Confidence & Threshold Cut-off -->
        <div style="border: 1px solid #e2e8f0; border-radius: 8px; padding: 14px; background: #ffffff; display: flex; flex-direction: column; justify-content: space-between;">
            <div>
                <div style="font-size: 14px; font-weight: 700; color: #0f172a; margin-bottom: 4px;">
                    1. Per-Article Confidence vs. Cut-off
                </div>
                <div style="font-size: 11.5px; color: #64748b; margin-bottom: 14px;">
                    Items with confidence &ge; threshold are auto-accepted; others go to human review.
                </div>
                <div id="panel1-bars" style="position: relative; padding-top: 8px; padding-bottom: 8px;"></div>
            </div>
            <div style="display: flex; gap: 12px; font-size: 11px; color: #334155; border-top: 1px solid #f1f5f9; padding-top: 10px; margin-top: 8px; flex-wrap: wrap;">
                <span><b style="color: #16a34a;">■</b> Auto-Accept (Correct AI)</span>
                <span><b style="color: #dc2626;">■</b> Auto-Accept (AI Error!)</span>
                <span><b style="color: #f59e0b;">■</b> Deferred to Human</span>
            </div>
        </div>

        <!-- PANEL 2: Effect of Routing Bar Chart -->
        <div style="border: 1px solid #e2e8f0; border-radius: 8px; padding: 14px; background: #ffffff; display: flex; flex-direction: column; justify-content: space-between;">
            <div>
                <div style="font-size: 14px; font-weight: 700; color: #0f172a; margin-bottom: 4px;">
                    2. Effect of Routing on Workload & Quality
                </div>
                <div style="font-size: 11.5px; color: #64748b; margin-bottom: 12px;">
                    100% AI (Gray) vs. Active Threshold Routing (Blue)
                </div>
                <div id="panel2-chart" style="display: flex; justify-content: space-around; align-items: flex-end; height: 215px; padding: 20px 6px 0 6px; border-bottom: 1px solid #cbd5e1;"></div>
                <div style="display: flex; justify-content: space-around; font-size: 11px; font-weight: 600; color: #334155; margin-top: 8px; text-align: center;">
                    <div style="width: 24%;">AI<br>Coverage</div>
                    <div style="width: 24%;">Human<br>Workload</div>
                    <div style="width: 24%;">Accepted<br>Error Risk</div>
                    <div style="width: 24%;">Final<br>Per-Label F1</div>
                </div>
            </div>
            <div style="display: flex; justify-content: center; gap: 16px; font-size: 11px; color: #334155; border-top: 1px solid #f1f5f9; padding-top: 10px; margin-top: 8px;">
                <span><b style="color: #94a3b8;">■</b> 100% AI (No Routing)</span>
                <span><b style="color: #2563eb;">■</b> Routed Hybrid System</span>
            </div>
        </div>

        <!-- PANEL 3: Visual Metrics Table -->
        <div style="border: 1px solid #e2e8f0; border-radius: 8px; padding: 14px; background: #ffffff; display: flex; flex-direction: column; justify-content: space-between;">
            <div>
                <div style="font-size: 14px; font-weight: 700; color: #0f172a; margin-bottom: 4px;">
                    3. Evaluation Metrics Breakdown (n=5)
                </div>
                <div style="font-size: 11.5px; color: #64748b; margin-bottom: 10px;">
                    Section 5.3 metrics: Baseline AI vs. Routed Hybrid output
                </div>
                <table style="width: 100%; border-collapse: collapse; font-size: 12px;">
                    <thead>
                        <tr style="background: #f1f5f9; color: #1e293b; text-align: left; border-bottom: 2px solid #cbd5e1;">
                            <th style="padding: 7px 6px;">Metric (EXAGGERATION)</th>
                            <th style="padding: 7px 6px; text-align: center;">100% AI</th>
                            <th style="padding: 7px 6px; text-align: center; color: #1d4ed8;">Routed</th>
                        </tr>
                    </thead>
                    <tbody id="panel3-table-body"></tbody>
                </table>
            </div>
            <div id="panel3-status" style="font-size: 11.5px; padding: 8px 10px; border-radius: 6px; margin-top: 8px; font-weight: 600; text-align: center;"></div>
        </div>

    </div>
</div>

<script>
(function() {{
    const items = {data_json};
    const slider = document.getElementById("thresh-slider");
    const threshLabel = document.getElementById("thresh-value");
    const playBtn = document.getElementById("play-btn");
    const panel1 = document.getElementById("panel1-bars");
    const panel2 = document.getElementById("panel2-chart");
    const panel3Body = document.getElementById("panel3-table-body");
    const panel3Status = document.getElementById("panel3-status");

    function computeMetrics(preds, golds) {{
        const n = golds.length;
        let correct = 0, tp = 0, fp = 0, fn = 0, tn = 0;
        for (let i = 0; i < n; i++) {{
            if (preds[i] === golds[i]) correct++;
            if (preds[i] === 1 && golds[i] === 1) tp++;
            if (preds[i] === 1 && golds[i] === 0) fp++;
            if (preds[i] === 0 && golds[i] === 1) fn++;
            if (preds[i] === 0 && golds[i] === 0) tn++;
        }}
        const microF1 = correct / n;
        const f1 = (2 * tp + fp + fn) > 0 ? (2 * tp) / (2 * tp + fp + fn) : 0.0;
        const posGoldTotal = tp + fn;
        const negGoldTotal = tn + fp;
        const posPredTotal = tp + fp;
        const negPredTotal = tn + fn;

        return {{
            microF1: microF1,
            f1: f1,
            posGoldAcc: posGoldTotal > 0 ? tp / posGoldTotal : 0,
            negGoldAcc: negGoldTotal > 0 ? tn / negGoldTotal : 0,
            posPredErr: posPredTotal > 0 ? fp / posPredTotal : 0,
            negPredErr: negPredTotal > 0 ? fn / negPredTotal : 0,
            tp, fp, fn, tn, posGoldTotal, negGoldTotal, posPredTotal, negPredTotal
        }};
    }}

    const golds = items.map(d => d.gold_label);
    const aiPreds = items.map(d => d.majority_decision);
    const baseStats = computeMetrics(aiPreds, golds);
    const baseRisk = 1.0 - baseStats.microF1;

    function render(threshold) {{
        threshLabel.textContent = threshold.toFixed(2);

        // Scale helper for Panel 1 [0.70 .. 1.00]
        const minX = 0.70, maxX = 1.00;
        const pct = Math.max(0, Math.min(100, ((threshold - minX) / (maxX - minX)) * 100));

        // 1. Render Panel 1 Bars + Vertical Threshold Line
        let p1Html = `
            <div style="position: relative;">
                <div style="position: absolute; left: ${{pct}}%; top: -6px; bottom: -6px; width: 2.5px; background: #1d4ed8; z-index: 10; transition: left 0.08s linear;">
                    <div style="position: absolute; top: -16px; left: -28px; background: #1d4ed8; color: white; font-size: 10px; font-weight: 700; padding: 1px 5px; border-radius: 4px;">
                        T=${{threshold.toFixed(2)}}
                    </div>
                </div>
        `;

        let acceptedCount = 0;
        let acceptedErrors = 0;
        const hybridPreds = [];

        items.forEach(item => {{
            const isAccepted = item.conf >= threshold;
            if (isAccepted) {{
                acceptedCount++;
                if (!item.is_correct) acceptedErrors++;
                hybridPreds.push(item.majority_decision);
            }} else {{
                hybridPreds.push(item.gold_label); // Human review fixes deferred item
            }}

            const barWidthPct = Math.max(5, Math.min(100, ((item.conf - minX) / (maxX - minX)) * 100));
            let barColor = "#16a34a"; // Green for accepted & correct
            let badgeText = "Auto-Accept (AI)";
            let badgeBg = "#dcfce7";
            let badgeColor = "#166534";

            if (isAccepted && !item.is_correct) {{
                barColor = "#dc2626"; // Red for accepted AI false alarm
                badgeText = "Auto-Accept (AI Error!)";
                badgeBg = "#fee2e2";
                badgeColor = "#991b1b";
            }} else if (!isAccepted) {{
                barColor = "#f59e0b"; // Amber for deferred to human
                badgeText = "Defer to Human";
                badgeBg = "#fef3c7";
                badgeColor = "#92400e";
            }}

            p1Html += `
                <div style="margin-bottom: 11px;">
                    <div style="display: flex; justify-content: space-between; font-size: 11.5px; margin-bottom: 3px;">
                        <span style="font-weight: 600;">Item ${{item.item_id}} <span style="color:#64748b; font-weight:400;">(Gold=${{item.gold_label}}, AI=${{item.majority_decision}})</span></span>
                        <span style="background: ${{badgeBg}}; color: ${{badgeColor}}; padding: 1px 6px; border-radius: 4px; font-weight: 700; font-size: 10.5px;">
                            Conf: ${{item.conf.toFixed(2)}} &rarr; ${{badgeText}}
                        </span>
                    </div>
                    <div style="width: 100%; background: #f1f5f9; height: 16px; border-radius: 4px; overflow: hidden; border: 1px solid #cbd5e1;">
                        <div style="width: ${{barWidthPct}}%; background: ${{barColor}}; height: 100%; transition: background-color 0.2s;"></div>
                    </div>
                </div>
            `;
        }});

        p1Html += `
                <div style="display: flex; justify-content: space-between; font-size: 10px; color: #64748b; margin-top: 4px;">
                    <span>0.70</span><span>0.80</span><span>0.85</span><span>0.90</span><span>0.95</span><span>1.00</span>
                </div>
            </div>
        `;
        panel1.innerHTML = p1Html;

        // 2. Render Panel 2 (Effect of Routing Chart)
        const coverage = acceptedCount / items.length;
        const workload = 1.0 - coverage;
        const routedRisk = acceptedCount > 0 ? acceptedErrors / acceptedCount : 0.0;
        const hybridStats = computeMetrics(hybridPreds, golds);

        const barsData = [
            {{ base: 1.0, routed: coverage, isPct: true }},
            {{ base: 0.0, routed: workload, isPct: true }},
            {{ base: baseRisk, routed: routedRisk, isPct: true }},
            {{ base: baseStats.f1, routed: hybridStats.f1, isPct: false }}
        ];

        let p2Html = "";
        barsData.forEach(b => {{
            const baseH = Math.round(b.base * 165);
            const routeH = Math.round(b.routed * 165);
            const baseLbl = b.isPct ? Math.round(b.base * 100) + "%" : b.base.toFixed(2);
            const routeLbl = b.isPct ? Math.round(b.routed * 100) + "%" : b.routed.toFixed(2);

            p2Html += `
                <div style="display: flex; align-items: flex-end; gap: 5px; width: 22%; justify-content: center;">
                    <div style="display: flex; flex-direction: column; align-items: center;">
                        <span style="font-size: 10.5px; color: #475569; margin-bottom: 3px;">${{baseLbl}}</span>
                        <div style="width: 24px; height: ${{Math.max(3, baseH)}}px; background: #94a3b8; border: 1px solid #475569; border-radius: 3px 3px 0 0;"></div>
                    </div>
                    <div style="display: flex; flex-direction: column; align-items: center;">
                        <span style="font-size: 11px; font-weight: 700; color: #1d4ed8; margin-bottom: 3px;">${{routeLbl}}</span>
                        <div style="width: 24px; height: ${{Math.max(3, routeH)}}px; background: #2563eb; border: 1px solid #1e3a8a; border-radius: 3px 3px 0 0; transition: height 0.15s;"></div>
                    </div>
                </div>
            `;
        }});
        panel2.innerHTML = p2Html;

        // 3. Render Panel 3 (Metrics Table)
        const tableRows = [
            ["Micro-F1 (Overall Accuracy)", `${{Math.round(baseStats.microF1 * 100)}}% (4/5)`, `${{Math.round(hybridStats.microF1 * 100)}}% (${{Math.round(hybridStats.microF1 * 5)}}/5)`],
            ["Per-Label F1 (EXAGGERATION)", baseStats.f1.toFixed(2), hybridStats.f1.toFixed(2)],
            ["Accuracy on Positive Gold (gold=1)", `${{Math.round(baseStats.posGoldAcc * 100)}}% (${{baseStats.tp}}/${{baseStats.posGoldTotal}})`, `${{Math.round(hybridStats.posGoldAcc * 100)}}% (${{hybridStats.tp}}/${{hybridStats.posGoldTotal}})`],
            ["Accuracy on Negative Gold (gold=0)", `${{Math.round(baseStats.negGoldAcc * 100)}}% (${{baseStats.tn}}/${{baseStats.negGoldTotal}})`, `${{Math.round(hybridStats.negGoldAcc * 100)}}% (${{hybridStats.tn}}/${{hybridStats.negGoldTotal}})`],
            ["False Alarm Rate (AI pred=1)", `${{Math.round(baseStats.posPredErr * 100)}}% (${{baseStats.fp}}/${{baseStats.posPredTotal}})`, `${{Math.round(routedRisk * 100)}}% (${{acceptedErrors}}/${{acceptedCount}})`],
            ["Error Rate on Negative Preds (pred=0)", `${{Math.round(baseStats.negPredErr * 100)}}% (0/1)`, "0%"]
        ];

        panel3Body.innerHTML = tableRows.map((r, idx) => `
            <tr style="border-bottom: 1px solid #e2e8f0; background: ${{idx % 2 === 0 ? '#ffffff' : '#f8fafc'}};">
                <td style="padding: 6.5px 6px; font-weight: 500; color: #334155;">${{r[0]}}</td>
                <td style="padding: 6.5px 6px; text-align: center; color: #64748b;">${{r[1]}}</td>
                <td style="padding: 6.5px 6px; text-align: center; font-weight: 700; color: #1d4ed8;">${{r[2]}}</td>
            </tr>
        `).join("");

        if (acceptedErrors === 0 && acceptedCount === 4) {{
            panel3Status.style.background = "#dcfce7";
            panel3Status.style.color = "#166534";
            panel3Status.textContent = "Optimal Operating Point: 80% AI Coverage with 0% AI Error Risk (F1 = 1.00)";
        }} else if (acceptedErrors > 0) {{
            panel3Status.style.background = "#fee2e2";
            panel3Status.style.color = "#991b1b";
            panel3Status.textContent = "Low Threshold Warning: False alarm (Item 4) accepted without human review!";
        }} else {{
            panel3Status.style.background = "#fef3c7";
            panel3Status.style.color = "#92400e";
            panel3Status.textContent = `Conservative Threshold: Correct AI decisions are being deferred (Workload = ${{Math.round(workload*100)}}%)`;
        }}
    }}

    // Auto-Play animation loop
    let isPlaying = true;
    let direction = 1;
    let currentVal = 0.81;

    const timer = setInterval(() => {{
        if (!isPlaying) return;
        currentVal = Math.round((currentVal + direction * 0.01) * 100) / 100;
        if (currentVal >= 0.97) direction = -1;
        if (currentVal <= 0.81) direction = 1;
        slider.value = currentVal;
        render(currentVal);
    }}, 350);

    playBtn.addEventListener("click", () => {{
        isPlaying = !isPlaying;
        playBtn.textContent = isPlaying ? "⏸ Pause Auto-Play" : "▶ Resume Auto-Play";
        playBtn.style.background = isPlaying ? "#1d4ed8" : "#059669";
    }});

    slider.addEventListener("input", (e) => {{
        if (isPlaying) {{
            isPlaying = false;
            playBtn.textContent = "▶ Resume Auto-Play";
            playBtn.style.background = "#059669";
        }}
        currentVal = parseFloat(e.target.value);
        render(currentVal);
    }});

    render(currentVal);
}})();
</script>
"""

display(HTML(html_template))

Metric (EXAGGERATION),100% AI,Routed


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from sklearn.metrics import f1_score, accuracy_score
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

df = pd.read_csv("../data/results/poc_gemma3_results.csv")

def plot_interactive_routing(threshold_int=90):
    threshold = threshold_int / 100.0
    
    y_true = df["gold_label"].astype(int).values
    y_pred = df["majority_decision"].astype(int).values
    y_correct = df["is_correct"].astype(bool).values
    confs = df["mean_verbalised_conf"].astype(float).values
    
    accepted = confs >= threshold
    coverage = accepted.mean()
    workload = (~accepted).mean()
    
    # Baseline vs. Routed metrics
    acc_base = accuracy_score(y_true, y_pred)
    f1_base = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    risk_base = 1.0 - acc_base
    
    risk_routed = (1.0 - y_correct[accepted].mean()) if np.any(accepted) else 0.0
    y_hybrid = np.where(accepted, y_pred, y_true)
    acc_hybrid = accuracy_score(y_true, y_hybrid)
    f1_hybrid = f1_score(y_true, y_hybrid, pos_label=1, zero_division=0)
    
    pos_gold_base = y_correct[y_true == 1].mean()
    neg_gold_base = y_correct[y_true == 0].mean()
    pos_gold_hyb = (y_hybrid[y_true == 1] == 1).mean()
    neg_gold_hyb = (y_hybrid[y_true == 0] == 0).mean()
    
    fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.0), gridspec_kw={"width_ratios": [1.25, 1.0, 1.15]})
    
    # Panel 1: Per-Article Confidence vs. Moving Threshold
    labels = [f"Item {r.item_id} (Gold={r.gold_label}, AI={r.majority_decision})" for _, r in df.iterrows()]
    bar_colors = [
        "#2ca02c" if (acc and corr) else ("#d62728" if (acc and not corr) else "#f39c12")
        for acc, corr in zip(accepted, y_correct)
    ]
    bars = axes[0].barh(labels, confs, color=bar_colors, height=0.52, edgecolor="black", alpha=0.9)
    axes[0].axvline(threshold, color="#1f77b4", linestyle="--", linewidth=2.5)
    
    for bar, acc in zip(bars, accepted):
        status = "Auto-Accept (AI)" if acc else "Defer to Human"
        axes[0].text(
            bar.get_width() + 0.006, bar.get_y() + bar.get_height()/2,
            f"{bar.get_width():.2f} -> {status}", va="center", fontsize=8.5, fontweight="bold"
        )
    axes[0].set_xlim(0.70, 1.14)
    axes[0].invert_yaxis()
    axes[0].set_title(f"1. Per-Article Cut-off (Threshold = {threshold:.2f})", fontsize=11, fontweight="bold")
    axes[0].set_xlabel("Mean Verbalised Confidence", fontsize=10)
    axes[0].legend(handles=[
        Patch(facecolor="#2ca02c", edgecolor="black", label="Accepted (Correct AI)"),
        Patch(facecolor="#d62728", edgecolor="black", label="Accepted (AI Error)"),
        Patch(facecolor="#f39c12", edgecolor="black", label="Deferred to Human"),
        Line2D([0], [0], color="#1f77b4", lw=2, ls="--", label=f"Threshold ({threshold:.2f})")
    ], loc="lower right", fontsize=8)
    
    # Panel 2: Effect of Routing Bar Chart
    m_names = ["AI Coverage", "Human Work", "AI Error Risk", "Final F1"]
    x = np.arange(len(m_names))
    w = 0.35
    b1 = axes[1].bar(x - w/2, [1.0, 0.0, risk_base, f1_base], w, label="100% AI", color="#95a5a6", edgecolor="black")
    b2 = axes[1].bar(x + w/2, [coverage, workload, risk_routed, f1_hybrid], w, label="Routed", color="#1f77b4", edgecolor="black")
    
    for i, r in enumerate(b2):
        val_str = f"{r.get_height():.0%}" if i < 3 else f"{r.get_height():.2f}"
        axes[1].text(r.get_x() + r.get_width()/2, r.get_height() + 0.02, val_str, ha="center", fontsize=9, fontweight="bold", color="#0d47a1")
    axes[1].set_ylim(0, 1.18)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(m_names, fontsize=9, fontweight="bold")
    axes[1].set_title("2. Effect on Workload & Quality", fontsize=11, fontweight="bold")
    axes[1].legend(loc="upper center", ncol=2, fontsize=8.5)
    
    # Panel 3: Visual Table of Metrics
    axes[2].axis("off")
    axes[2].set_title("3. Summary Metrics Table (n=5)", fontsize=11, fontweight="bold")
    table_data = [
        ["Micro-F1 (Accuracy)", f"{acc_base:.0%} (4/5)", f"{acc_hybrid:.0%}"],
        ["Per-Label F1 (EXAGGERATION)", f"{f1_base:.2f}", f"{f1_hybrid:.2f}"],
        ["Acc. on Positive Gold (gold=1)", f"{pos_gold_base:.0%} (3/3)", f"{pos_gold_hyb:.0%}"],
        ["Acc. on Negative Gold (gold=0)", f"{neg_gold_base:.0%} (1/2)", f"{neg_gold_hyb:.0%}"],
        ["Accepted AI Error Risk", f"{risk_base:.0%} (1/5)", f"{risk_routed:.0%}"],
        ["Human Review Workload", "0% (0/5)", f"{workload:.0%} ({int((~accepted).sum())}/5)"]
    ]
    tbl = axes[2].table(
        cellText=table_data,
        colLabels=["Metric", "100% AI", f"Routed (>={threshold:.2f})"],
        loc="center", cellLoc="center", colWidths=[0.52, 0.22, 0.26]
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1.0, 1.65)
    
    plt.tight_layout()
    plt.show()

play = widgets.Play(value=82, min=80, max=98, step=1, interval=450, description="Auto-Play")
slider = widgets.IntSlider(value=82, min=80, max=98, step=1, description="Threshold %")
widgets.jslink((play, "value"), (slider, "value"))

ui = widgets.HBox([play, slider])
out = widgets.interactive_output(plot_interactive_routing, {"threshold_int": slider})
display(ui, out)

Output()